# DATA MODELING

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
DATA_FILE = "cleaned_agri_crop_yield_dataset.csv"
OUT = "model_output"
CHARTS = os.path.join(OUT, "charts")
os.makedirs(CHARTS, exist_ok=True)

In [3]:
df = pd.read_csv(DATA_FILE)
df

,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield,Area_Normalized,Annual_Rainfall_Normalized,Fertilizer_Normalized,Pesticide_Normalized,Yield_Normalized
0,Millet,2020,Kharif,Chhattisgarh,15076.5000,17031.9123,987.8,2.188493e+06,75230.8098,1.1297,0.3694,0.3421,0.4665,0.4581,0.1009
1,Sugarcane,2001,Rabi,Madhya Pradesh,6047.0700,145774.8398,1760.6,3.185354e+05,47011.8808,6.8491,0.1451,0.7265,0.0659,0.2854,1.0000
2,Barley,2015,Kharif,Chhattisgarh,4505.9600,10996.0146,1294.2,3.318065e+05,8451.0804,2.4403,0.1068,0.4945,0.0688,0.0495,0.3069
3,Soybean,1999,Kharif,Andhra Pradesh,741.3000,749.3950,300.0,9.602865e+04,1146.2010,1.0109,0.0133,0.0000,0.0183,0.0048,0.0822
4,Cotton,2014,Kharif,Madhya Pradesh,2604.4300,3865.8288,414.5,2.846358e+05,16639.2337,1.4843,0.0596,0.0570,0.0587,0.0996,0.1566
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,Millet,2008,Summer,Telangana,4336.2000,7803.2247,973.6,4.686581e+05,2398.8899,1.7996,0.1026,0.3351,0.0981,0.0125,0.2062
1196,Rice,2012,Rabi,Telangana,4434.2200,15669.3493,1544.5,5.334346e+05,24741.8619,3.5337,0.1050,0.6191,0.1120,0.1492,0.4788
1197,Soybean,2010,Kharif,Rajasthan,8484.3500,11892.9253,300.0,1.476120e+06,45101.3348,1.4017,0.2056,0.0000,0.3139,0.2737,0.1436
1198,Maize,2004,Summer,Maharashtra,3329.8200,10405.8582,877.0,5.312131e+05,10709.9066,3.1251,0.0776,0.2870,0.1115,0.0633,0.4146


In [4]:
if "Yield" not in df.columns:
    raise ValueError("The dataset must contain a 'Yield' column.")

df = df.dropna(subset=["Yield"]).reset_index(drop=True)

# Prevent target leakage: Yield and Yield_Normalized are not predictors.
excluded = [c for c in ["Yield", "Yield_Normalized"] if c in df.columns]
# Also remove normalized copies of input variables to avoid duplicated information.
excluded += [c for c in df.columns if c.endswith("_Normalized") and c != "Yield_Normalized"]

In [5]:
features = [c for c in df.columns if c not in excluded]
X, y = df[features], df["Yield"]

cat = X.select_dtypes(include=["object", "category"]).columns.tolist()
num = X.select_dtypes(include=np.number).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer([
    ("categorical", encoder, cat),
    ("numeric", "passthrough", num)
])

In [6]:
features = [c for c in df.columns if c not in excluded]
X, y = df[features], df["Yield"]

cat = X.select_dtypes(include=["object", "category"]).columns.tolist()
num = X.select_dtypes(include=np.number).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer([
    ("categorical", encoder, cat),
    ("numeric", "passthrough", num)
])

In [7]:
model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_features="sqrt"
)

In [8]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, pred)
mse = mean_squared_error(y_test, pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, pred)

In [9]:
print("\n===== RANDOM FOREST REGRESSION RESULTS =====")
print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")


===== RANDOM FOREST REGRESSION RESULTS =====
MAE  : 0.3763
MSE  : 0.3863
RMSE : 0.6216
R2   : 0.9127


In [10]:
pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2"],
    "Value": [mae, mse, rmse, r2]
}).to_csv(os.path.join(OUT, "model_evaluation_metrics.csv"), index=False)

In [11]:
predictions = X_test.copy()
predictions["Actual_Yield"] = y_test.values
predictions["Predicted_Yield"] = pred
predictions["Absolute_Error"] = np.abs(y_test.values - pred)
predictions.to_csv(os.path.join(OUT, "test_predictions.csv"), index=False)

In [12]:
names = pipeline.named_steps["preprocessor"].get_feature_names_out()
importance = pipeline.named_steps["model"].feature_importances_
fi = pd.DataFrame({"Feature": names, "Importance": importance}).sort_values(
    "Importance", ascending=False
)
fi.to_csv(os.path.join(OUT, "feature_importance.csv"), index=False)

In [13]:
plt.figure(figsize=(8,6))
plt.scatter(y_test, pred, alpha=.7)
lo, hi = min(y_test.min(), pred.min()), max(y_test.max(), pred.max())
plt.plot([lo,hi],[lo,hi],"--")
plt.xlabel("Actual Yield"); plt.ylabel("Predicted Yield")
plt.title("Actual vs Predicted Yield"); plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "actual_vs_predicted.png"), dpi=180)
plt.close()

In [14]:
res = y_test.values - pred
plt.figure(figsize=(8,6))
plt.scatter(pred, res, alpha=.7)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Yield"); plt.ylabel("Residual")
plt.title("Residual Plot"); plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "residual_plot.png"), dpi=180)
plt.close()

In [15]:
top = fi.head(15).sort_values("Importance")
plt.figure(figsize=(9,6))
plt.barh(top["Feature"], top["Importance"])
plt.xlabel("Importance"); plt.ylabel("Feature")
plt.title("Top Feature Importances"); plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "feature_importance.png"), dpi=180)
plt.close()

In [16]:
print("\nAll outputs saved in:", OUT)


All outputs saved in: model_output
